# YouTube Channel Analysis

This notebook connects to the PostgreSQL container, loads the `channel_stats` table, and produces a ranked subscriber chart plus derived channel metrics.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (12, 7), "axes.titlesize": 16, "axes.labelsize": 12})

In [ ]:
db_config = {
    "host": os.getenv("POSTGRES_HOST", "postgres"),
    "port": int(os.getenv("POSTGRES_PORT", "5432")),
    "database": os.getenv("POSTGRES_DB", "youtube_data"),
    "user": os.getenv("POSTGRES_USER", "airflow"),
    "password": os.getenv("POSTGRES_PASSWORD", "change_me_strong_password"),
}

query = """
SELECT channel_title, subscriber_count, total_views, video_count
FROM channel_stats
ORDER BY subscriber_count DESC;
"""

connection = psycopg2.connect(**db_config)
channel_df = pd.read_sql_query(query, connection)
connection.close()

channel_df.head()

In [ ]:
analysis_df = channel_df.copy()
analysis_df["avg_views_per_video"] = analysis_df["total_views"].div(analysis_df["video_count"].replace({0: pd.NA}))
analysis_df["engagement_ratio"] = analysis_df["subscriber_count"].div(analysis_df["total_views"].replace({0: pd.NA}))

analysis_df[["channel_title", "avg_views_per_video", "engagement_ratio"]].head()

In [ ]:
top_channels = analysis_df.dropna(subset=["subscriber_count"]).copy()
top_channels["channel_title"] = top_channels["channel_title"].fillna("Unknown Channel")
top_channels = top_channels.head(15)

plt.figure(figsize=(14, 8))
sns.barplot(data=top_channels, x="subscriber_count", y="channel_title", palette="Blues_r")
plt.title("YouTube Channels by Subscriber Count")
plt.xlabel("Subscriber Count")
plt.ylabel("Channel Title")
plt.tight_layout()
plt.show()

In [ ]:
summary = analysis_df.agg({
    "subscriber_count": ["mean", "median", "max"],
    "total_views": ["mean", "median", "max"],
    "video_count": ["mean", "median", "max"],
    "avg_views_per_video": ["mean", "median", "max"],
    "engagement_ratio": ["mean", "median", "max"],
})

summary

In [ ]:
analysis_df.sort_values("avg_views_per_video", ascending=False)[["channel_title", "avg_views_per_video", "engagement_ratio"]].head(10)